# 99 — Laporan

**Tujuan:** menyatukan data, metode, hasil validation/test, error analysis, dan keterbatasan dalam satu dokumen yang bisa dibaca orang lain.

**Prasyarat:** notebook `01`–`06` sudah dijalankan sesuai urutan.

**Estimasi waktu:** 1 menit (membaca artefak, tanpa training).

## Glosarium

- **Validation**: data untuk memilih model dan threshold.
- **Test**: data untuk laporan akhir, dievaluasi sekali.
- **F1 bully**: metrik utama; precision/recall menjelaskan error.
- **Reproducibility**: hasil bisa diulang dengan seed dan versi yang sama.

## 1. Artefak yang tersedia

Laporan hanya menyatakan hasil yang benar-benar ada. Kalau file belum ada,
notebook akan mengatakan belum final, bukan mengisi angka nol. Ini menjaga
laporan tetap jujur saat eksperimen belum selesai.

In [ ]:
import json

import pandas as pd

from cyberbully.config import DATA_PROCESSED, LABEL_COLUMN, MODELS_DIR, OUT_DIR

artifacts = {
    "baseline_val": OUT_DIR / "baseline_val.json",
    "baseline_test": OUT_DIR / "baseline_test.json",
    "threshold_a": OUT_DIR / "model_a" / "threshold.json",
    "threshold_b": OUT_DIR / "model_b" / "threshold.json",
    "indobert_a_val": MODELS_DIR / "indobert_real" / "val_metrics.json",
    "indobert_b_val": MODELS_DIR / "indobert_aug" / "val_metrics.json",
    "indobert_a_test": OUT_DIR / "model_a" / "test_metrics.json",
    "indobert_b_test": OUT_DIR / "model_b" / "test_metrics.json",
}
print({name: path.exists() for name, path in artifacts.items()})


## 2. Data dan split

Dataset berisi komentar Twitter Indonesia yang sudah dianonimkan penulisnya
(`USER`/`URL`). Label bully = `HS` atau `Abusive` bernilai 1. Deduplikasi
dilakukan sebelum split agar tidak ada teks yang muncul di train dan test.

In [ ]:
splits = {}
for name in ("train", "val", "test"):
    path = DATA_PROCESSED / f"{name}.csv"
    if path.exists():
        frame = pd.read_csv(path)
        splits[name] = {"baris": len(frame), "positif": int(frame[LABEL_COLUMN].sum())}
print(pd.DataFrame(splits).T.to_string() if splits else "Split belum tersedia.")

## 3. Hasil validation

Tabel membandingkan baseline dan IndoBERT pada validation, dengan threshold
masing-masing. Angka ini dipakai untuk memilih model; klaim akhir tetap menunggu
hasil test.

In [ ]:
rows = []
if artifacts["baseline_val"].exists():
    payload = json.loads(artifacts["baseline_val"].read_text())
    rows.append({"model": "TF-IDF + LogReg", "threshold": payload["threshold"], **{k: payload["val"][k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
for key, label in (("a", "IndoBERT A — real"), ("b", "IndoBERT B — asli+sintetis")):
    path = artifacts[f"indobert_{key}_val"]
    if path.exists():
        payload = json.loads(path.read_text())
        rows.append({"model": label, "threshold": payload["threshold"], **{k: payload[k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
print(pd.DataFrame(rows).set_index("model").round(3).to_string() if rows else "Belum ada hasil validation.")


## 4. Hasil test (sekali)

Test hanya dievaluasi setelah model dan threshold dibekukan. Kalau hasil test
belum ada, laporan ini belum final. Jangan mengulang test untuk mengejar angka
yang lebih bagus.

In [ ]:
test_rows = []
if artifacts["baseline_test"].exists():
    payload = json.loads(artifacts["baseline_test"].read_text())
    test_rows.append({"model": "TF-IDF + LogReg", "threshold": payload["threshold"], **{k: payload["metrics"][k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
for key, label in (("a", "IndoBERT A — real"), ("b", "IndoBERT B — asli+sintetis")):
    path = artifacts[f"indobert_{key}_test"]
    if path.exists():
        payload = json.loads(path.read_text())
        test_rows.append({"model": label, "threshold": payload["threshold"], **{k: payload[k] for k in ("accuracy", "precision_bully", "recall_bully", "f1_bully")}})
print(pd.DataFrame(test_rows).set_index("model").round(3).to_string() if test_rows else "Hasil test belum ada; laporan belum final.")


## 5. Ringkasan error

Bagian ini melengkapi tabel dengan jumlah false negative dan false positive pada
validation. Detail contoh dan subgroup ada di notebook `03`. Angka ini membantu
membaca apakah model lebih sering melewatkan bully atau menuduh teks biasa.

In [ ]:
for key, label in (("a", "Model A — real"), ("b", "Model B — asli+sintetis")):
    pred_path = MODELS_DIR / ("indobert_real" if key == "a" else "indobert_aug") / "val_predictions.csv"
    threshold_path = artifacts[f"threshold_{key}"]
    if not pred_path.exists():
        print(label, ": prediksi belum ada")
        continue
    frame = pd.read_csv(pred_path)
    threshold = 0.5
    if threshold_path.exists():
        threshold = float(json.loads(threshold_path.read_text())["threshold"])
    prediction = (frame["probability"] >= threshold).astype(int)
    fn = int(((frame[LABEL_COLUMN] == 1) & (prediction == 0)).sum())
    fp = int(((frame[LABEL_COLUMN] == 0) & (prediction == 1)).sum())
    print(f"{label}: threshold {threshold:.2f} | false negative {fn} | false positive {fp}")


## 6. Metode, keterbatasan, dan etika

Normalisasi minimal mempertahankan emoji/tanda baca; dedup dilakukan sebelum
split; baseline memakai TF-IDF 1–2 gram; IndoBERT di-fine-tune dengan class
weights dan checkpoint terbaik berdasarkan F1 validation. Model tidak membaca
konteks percakapan atau hubungan antar pengguna, sehingga sindiran halus dan kata
kasar akrab bisa salah dibaca. Dataset CC BY-NC-SA 4.0: non-komersial, atribusi,
share-alike. Model adalah alat bantu moderasi, bukan penghukum otomatis.

## Ringkasan

Laporan ini hanya memuat hasil yang ada di artefak. Validation dipakai untuk
memilih model dan threshold; test dipakai sekali untuk klaim akhir. Bagian yang
kosong berarti eksperimennya belum dijalankan, bukan hasil nol.